# Batch Inference → Gold Layer (species_predictions)

Predict species for **all** res-7 hexes in Spain and write to S3 gold layer.

**Output path:** `s3://ie-datalake/gold/species_predictions/country=ES/snapshot=YYYY-MM-DD/h3_resolution=7/`

**Schema (long format, one row per hex–species):**

| Column | Type | Description |
|--------|------|-------------|
| h3_index | str | H3 cell ID |
| species_id | int | GBIF taxon key |
| species_name | str | Scientific name |
| probability | float | 0–1, model output |
| rank | int | 1-based, top predicted first |
| is_threatened | bool | From target species |
| is_invasive | bool | From target species |

**Requires:** Run `species_predictor.ipynb` first to produce model + scaler.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════════

COUNTRY = "ES"
H3_RES = 7
S3_BUCKET = "ie-datalake"
GOLD_OSM_HEX = "gold/osm_hex_features"
GOLD_GEE_TERRAIN = "gold/gee_hex_terrain"
GOLD_CELL_METRICS = "gold/gbif_cell_metrics"
GOLD_NATURE2000 = "gold/nature2000_cell_protection"
GEE_TERRAIN_SNAPSHOT = "2019"
NATURE2000_SNAPSHOT_DATE = "2026-02-27"
AWS_PROFILE = "486717354268_PowerUserAccess"

# Gold output
GOLD_SPECIES_PREDICTIONS = "gold/species_predictions"
SNAPSHOT = "2024-03-05"  # inference run date; change per run

OUTPUT_DIR = "output"
MODEL_PATH = f"{OUTPUT_DIR}/species_predictor.pt"
XGB_MODEL_PATH = f"{OUTPUT_DIR}/species_predictor_xgb.pkl"
TARGET_SPECIES_PATH = f"{OUTPUT_DIR}/target_species.csv"
SCALER_PATH = f"{OUTPUT_DIR}/scaler.pkl"
USE_XGBOOST = True  # XGBoost is best; set False for MLP
HIDDEN_SIZES = [256, 128]
BATCH_SIZE = 2048
TOP_N_PER_HEX = 20

In [ ]:
%pip install -q pandas numpy scikit-learn torch h3 pyarrow boto3 s3fs xgboost

In [ ]:
import io
import os
import re
from pathlib import Path

import h3
import numpy as np
import pandas as pd
import torch
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.fs as pafs
import pyarrow.parquet as pq
import boto3
import s3fs

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

## 1) S3 connection

In [ ]:
session = boto3.Session(profile_name=AWS_PROFILE)
creds = session.get_credentials().get_frozen_credentials()
region = session.region_name or "eu-west-2"

fs_read = pafs.S3FileSystem(
    access_key=creds.access_key,
    secret_key=creds.secret_key,
    session_token=creds.token,
    region=region,
)

fs_write = s3fs.S3FileSystem(profile=AWS_PROFILE)
print(f"S3 ready (profile={AWS_PROFILE})")

## 2) Load all hexes from GEE terrain

In [ ]:
def _read_parquet(path: str) -> pd.DataFrame:
    """Read Parquet from S3."""
    tbl = ds.dataset(path, filesystem=fs_read, format="parquet").scanner().to_table()
    return tbl.to_pandas()

path_gee = f"{S3_BUCKET}/{GOLD_GEE_TERRAIN}/country={COUNTRY}/snapshot={GEE_TERRAIN_SNAPSHOT}/h3_resolution={H3_RES}"
df_gee = _read_parquet(path_gee)
h3_col = "h3_index" if "h3_index" in df_gee.columns else "h3_id"
df_hexes = df_gee[[h3_col]].drop_duplicates().rename(columns={h3_col: "h3_index"})
print(f"Hexes: {len(df_hexes)}")

## 3) Build features (OSM, GEE, cell metrics, Nature2000; hist=0 for inference hexes)

In [ ]:
h3_col = "h3_index"
df = df_hexes.copy()

# OSM
path_osm = f"{S3_BUCKET}/{GOLD_OSM_HEX}/country={COUNTRY}/h3_resolution={H3_RES}"
df_osm = _read_parquet(path_osm)
rc = "h3_index" if "h3_index" in df_osm.columns else "h3_id"
df = df.merge(df_osm, left_on=h3_col, right_on=rc, how="left")
if rc != h3_col and rc in df.columns:
    df = df.drop(columns=[rc])

# Nature2000
path_n2k = f"{S3_BUCKET}/{GOLD_NATURE2000}/country={COUNTRY}/h3_resolution={H3_RES}/snapshot_date={NATURE2000_SNAPSHOT_DATE}"
try:
    df_n2k = _read_parquet(path_n2k)
    nc = "h3_index" if "h3_index" in df_n2k.columns else "h3_id"
    n2k = df_n2k[[nc, "is_protected_area", "nearest_protected_distance"]].rename(columns={nc: h3_col})
    n2k["is_protected_area"] = (n2k["is_protected_area"].astype(str).str.lower() == "yes").astype(np.float32)
    n2k["nearest_protected_distance"] = pd.to_numeric(n2k["nearest_protected_distance"], errors="coerce").fillna(-1).astype(np.float32)
    df = df.merge(n2k, on=h3_col, how="left")
except Exception as e:
    print(f"Skip Nature2000: {e}")

# GEE terrain (elevation, slope, lc_*)
gee_cols = [c for c in df_gee.columns if c not in (h3_col, "h3_id", "h3_resolution")]
gee_col = "h3_index" if "h3_index" in df_gee.columns else "h3_id"
df_gee_sub = df_gee[[gee_col] + gee_cols].rename(columns={gee_col: h3_col})
df = df.merge(df_gee_sub, on=h3_col, how="left")

# Cell metrics (2019-2023, same as training to avoid leakage)
try:
    parts = []
    for year in range(2019, 2024):
        path_cell = f"{S3_BUCKET}/{GOLD_CELL_METRICS}/country={COUNTRY}/year={year}/h3_resolution={H3_RES}"
        try:
            parts.append(_read_parquet(path_cell))
        except Exception:
            pass
    df_cell = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    obs_col = "observation_count" if "observation_count" in df_cell.columns else "obs_count_all"
    cc = "h3_index" if "h3_index" in df_cell.columns else "h3_id"
    agg = df_cell.groupby(cc).agg({obs_col: "sum", **({"dqi": "mean"} if "dqi" in df_cell.columns else {})}).reset_index().rename(columns={cc: h3_col})
    if obs_col in agg.columns:
        agg["log_obs_count"] = np.log1p(agg[obs_col].fillna(0))
    df = df.merge(agg, on=h3_col, how="left")
except Exception as e:
    print(f"Skip cell metrics: {e}")

# neighbor_log_obs_mean
if "log_obs_count" in df.columns:
    obs_map = dict(zip(df[h3_col], df["log_obs_count"].fillna(0)))
    def neighbor_mean(h, m):
        try:
            nbs = list(h3.grid_disk(h, 1))
            vals = [m.get(n, 0) for n in nbs if n != h]
            return np.mean(vals) if vals else 0.0
        except Exception:
            return 0.0
    df["neighbor_log_obs_mean"] = df[h3_col].map(lambda h: neighbor_mean(h, obs_map))

# Historical presence (in_hex_last5y, in_k1_last5y, in_k2_last5y): missing cols added as 0 in step 4
df_target = pd.read_csv(TARGET_SPECIES_PATH)

print(f"Features: {len(df.columns)} cols, {len(df)} rows")

## 4) Load model, scaler, align feature cols

In [ ]:
import pickle

with open(SCALER_PATH, "rb") as f:
    obj = pickle.load(f)
scaler = obj["scaler"]
feat_cols = obj.get("feature_cols", [])

for c in feat_cols:
    if c not in df.columns:
        df[c] = 0.0
feat_cols = [c for c in feat_cols if c in df.columns]

X = df[feat_cols].fillna(0).replace([np.inf, -np.inf], 0)
X = np.clip(X.values.astype(np.float64), -1e15, 1e15).astype(np.float32)
X = scaler.transform(X)

if USE_XGBOOST and Path(XGB_MODEL_PATH).exists():
    with open(XGB_MODEL_PATH, "rb") as f:
        xgb_obj = pickle.load(f)
    xgb_multi = xgb_obj["model"]
    print(f"Loaded XGBoost from {XGB_MODEL_PATH}, {len(feat_cols)} features, {len(df_target)} species")
else:
    if USE_XGBOOST:
        print("XGB_MODEL_PATH not found, falling back to MLP")
        USE_XGBOOST = False
    
    class MultiLabelMLP(torch.nn.Module):
        def __init__(self, in_dim, out_dim, hidden_sizes=(256, 128), dropout=0.3):
            super().__init__()
            layers = []
            prev = in_dim
            for h in hidden_sizes:
                layers.extend([torch.nn.Linear(prev, h), torch.nn.ReLU(), torch.nn.Dropout(dropout)])
                prev = h
            layers.append(torch.nn.Linear(prev, out_dim))
            self.net = torch.nn.Sequential(*layers)
        def forward(self, x):
            return self.net(x)
    ckpt = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
    model = MultiLabelMLP(len(feat_cols), len(df_target), hidden_sizes=HIDDEN_SIZES, dropout=0.0)
    model.load_state_dict(ckpt["model"])
    model.eval()
    print(f"Loaded MLP from {MODEL_PATH}, {len(feat_cols)} features, {len(df_target)} species")

## 5) Batch inference → long-format DataFrame

In [ ]:
rows = []
hex_ids = df[h3_col].values

for start in range(0, len(X), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(X))
    x_batch = X[start:end]
    if USE_XGBOOST:
        probs = np.column_stack([p[:, 1] for p in xgb_multi.predict_proba(x_batch)]).astype(np.float32)
    else:
        x_t = torch.tensor(x_batch, dtype=torch.float32)
        with torch.no_grad():
            probs = torch.sigmoid(model(x_t)).numpy()
    batch_hexes = hex_ids[start:end]
    for i, h in enumerate(batch_hexes):
        top_idx = np.argsort(-probs[i])[:TOP_N_PER_HEX]
        for rank, j in enumerate(top_idx, 1):
            p = float(probs[i, j])
            if p < 0.01:
                continue
            r = df_target.iloc[j]
            rows.append({
                "h3_index": h,
                "species_id": int(r["species_id"]),
                "species_name": str(r["species_name"]),
                "probability": round(p, 6),
                "rank": rank,
                "is_threatened": bool(r.get("is_threatened", False)),
                "is_invasive": bool(r.get("is_invasive", False)),
            })
    print(f"Batch {start}–{end} / {len(X)}")

df_gold = pd.DataFrame(rows)
print(f"Gold rows: {len(df_gold)}")

## 6) Write to S3 gold layer

In [ ]:
out_path = f"{S3_BUCKET}/{GOLD_SPECIES_PREDICTIONS}/country={COUNTRY}/snapshot={SNAPSHOT}/h3_resolution={H3_RES}"
out_file = f"{out_path}/data.parquet"

# Write via s3fs
# Note: partition cols can be in data or inferred from path; DuckDB/Athena use path

buf = io.BytesIO()
df_gold.to_parquet(buf, index=False)
buf.seek(0)

fs_write.put(buf, out_file)
print(f"Written: s3://{out_file}")